# Tech Stock Return Prediction and Ranking System

## Team: Algorithm Architects

| Name | GitHub ID |
|------|-----------|
| Shivani Kankatala | ShivaniKankatala |
| Poorvi Nidsoshi | poorvinidsoshi01 |
| Priyanka Pawar | ppawar03-byte |

**Course:** IST707 — Applied Machine Learning, Syracuse University  
**Date:** May 5, 2026

> **Data:** All stock CSVs are available for download at: https://drive.google.com/drive/folders/1c4E3K-YjxeZGyBOpFGXVtexh7Qa1yCbv  
> Data source: Yahoo Finance via `yfinance` (adjusted for splits and dividends)

---
## 1. Introduction

College students entering the stock market for the first time face a uniquely difficult challenge: they want to start investing but lack the financial literacy, tools, and experience to make confident, data-driven decisions. Professional investment platforms are expensive or require prior knowledge to navigate effectively. Free tools like Robinhood provide only qualitative Buy/Hold/Sell ratings from analysts — they do not tell a first-time investor *which* stock is the best opportunity *this week*, *how much* they might gain or lose, or *what to actually do* with a small starting budget.

Our primary stakeholder is a **college student beginning to invest**, managing a small initial portfolio (e.g., $500–$2,000) with limited financial background. The core need is simple and practical: given a set of well-known stocks, which one should I buy this week, and what return can I reasonably expect? The student does not have time to read earnings reports or monitor markets daily — they need a weekly signal that is easy to act on and grounded in data.

This project builds a machine-learning-based stock ranking and signal system that predicts weekly returns for 15 well-known stocks across technology, finance, healthcare, energy, and consumer sectors. For each week, the system ranks all stocks by predicted 7-day return, generates a clear buy/avoid signal, and simulates portfolio performance to show whether following the model's signals would have been financially beneficial. The models used — ARIMA, XGBoost, LSTM, Elastic Net Logistic Regression, and Random Forest — are trained on historical price and volume data from 2015–2026 and evaluated on held-out data from 2024–2026.

The system directly addresses the student investor's need: rather than overwhelming them with raw data or complex analysis, it outputs a simple ranked list and a buy/avoid decision each week. Even modest accuracy above a random baseline translates into a meaningful edge for a beginner building their first portfolio.

---
## 2. Literature Review

### 2.1 Prior Work and Stakeholder Context

Stock return prediction has a long history in both academic research and industry practice. Fischer & Krauss (2018) demonstrated that LSTM networks applied to the S&P 500 achieve roughly 55–60% directional accuracy — a meaningful edge over a 50/50 baseline. Sezer et al. (2020) reviewed deep learning approaches to financial time series and confirmed that technical indicators combined with sequence models generally outperform simpler baselines on short-horizon return prediction. Pinelis & Ruppert (2022) showed that even professional quantitative models with 11 macroeconomic features achieve only ~64% directional accuracy on weekly stock prediction, establishing a realistic upper bound for what technical-feature-only models can achieve.

Student and first-time investors are a growing demographic: a 2021 FINRA survey found that younger investors (18–34) represented the fastest-growing segment of new brokerage account openings, yet reported the lowest levels of financial literacy. Platforms like Robinhood and Webull have lowered the barrier to entry, but their tools remain qualitative — analyst consensus ratings ("Buy"/"Hold") without quantitative predictions, weekly updates, or expected return estimates. A Schwab survey found that 67% of active traders report difficulty identifying the best stock picks; this problem is even more acute for students who have no prior experience to draw on.

Our system addresses this gap directly: it produces a concrete weekly ranking and buy/avoid signal that a student can act on without needing to understand the underlying model.

### 2.2 Why These Methods?

We chose five complementary model families to capture different aspects of return predictability:

- **ARIMA/ARIMAX:** Classical statistical baseline for time series. Models autocorrelation in returns; with exogenous variables captures short-term momentum. Appropriate as a baseline because it is well-understood and interpretable.
- **XGBoost:** Gradient-boosted decision tree model used as a **multiclass classifier** predicting three movement classes: Down, Neutral, and Up. An earlier version used `XGBRegressor` and converted predicted returns to directions using a zero cutoff — this was corrected because very small predicted returns close to zero were being treated as meaningful signals. The corrected approach uses a ±1% threshold so near-zero returns are classified as Neutral. XGBoost captures nonlinear relationships between lagged returns, moving averages, volatility, RSI, and market index features.
- **Elastic Net Logistic Regression:** Binary classification model (positive vs. negative return). Combines L1 and L2 regularization for correlated financial features. Produces the most interpretable output — a direct buy/avoid signal.
- **LSTM:** Recurrent neural network trained as a 3-class classifier (Up/Neutral/Down) using sector-based groupings — one model per sector — to capture shared dynamics within Tech, Finance, Healthcare, and Energy stocks. Balanced class weights prevent majority-class bias.
- **Random Forest:** Ensemble baseline combining multiple decision trees to reduce variance. Provides a strong comparison point against more complex models.

### 2.3 The Gap We Fill

No existing free tool ranks multiple stocks simultaneously by predicted movement strength, translates those rankings into buy/avoid decisions for a beginner, and shows the estimated financial outcome of following those decisions. This project fills that gap.

---
## 3. Data and Methods

### 3.1 Data

- **Source:** Yahoo Finance via `yfinance` (licensed NYSE/Nasdaq partner; adjusted for splits and dividends)
- **Download link:** https://drive.google.com/drive/folders/1c4E3K-YjxeZGyBOpFGXVtexh7Qa1yCbv
- **Universe:** 15 stocks, ~2,500 daily observations per stock (January 2015 – February 2026), resampled to **weekly frequency** (Friday close) → approximately **575 weekly rows per stock**, **8,625 total rows** across all stocks
- **Features per stock:** 15 engineered features (MA ratios, RSI, MACD, volatility, volume ratio, HL range, lagged returns)
- **Market Indices:** S&P 500 (^GSPC), Nasdaq (^IXIC) used as exogenous features

| Sector | Stocks | Avg Weekly Rows |
|--------|--------|-----------------|
| Technology | AAPL, MSFT, GOOGL, AMZN, NVDA, META, TSLA | ~575 |
| Finance | JPM, GS | ~575 |
| Healthcare | JNJ, PFE | ~575 |
| Energy | XOM, CVX | ~575 |
| Consumer / Industrial | WMT, BA | ~575 |

**Class balance (% of weeks with positive return):**

| Stock | % Up weeks | Notes |
|-------|-----------|-------|
| NVDA | ~58% | Highest — strong bull trend 2015–2026 |
| TSLA | ~55% | High volatility, frequent large moves |
| AAPL | ~54% | Steady uptrend |
| BA   | ~51% | Lowest — impacted by 737 MAX crisis and COVID |
| WMT  | ~53% | Stable, low volatility |

A trivial "always predict up" baseline would score ~51–58% depending on the stock, which is why naive baselines are included in evaluation.

**Key EDA Observations:**
- Weekly returns are approximately normally distributed with slight positive skew, centered near 0
- Stocks are positively correlated; tech stocks especially so; magnitude differences across weeks are sufficient to distinguish ranking positions
- NVDA shows the highest annualized volatility (~45%) and highest average weekly return; WMT and JNJ show the lowest
- No missing trading days found across all 15 stocks

> **EDA visualizations** (return distributions, correlation heatmap, volatility comparison, class balance chart) are available in `work/EDA/notebooks/`

### 3.2 Preprocessing and Feature Engineering

All data was loaded from CSV files, date columns converted to datetime, and price/volume columns cleaned of comma-formatting. Daily data was resampled to weekly frequency using Friday closing prices. The following feature families were constructed:

- **Lagged return features** — capture recent stock movement (lags 1, 2, 3, 5 days)
- **Moving average ratios** (5, 10, 20-day) — short and long-term price trend relative to current price
- **Rolling volatility** (5, 10-day) — recent price risk and fluctuation
- **Volume ratio** — current volume vs. 5-day average
- **RSI (14-day)** — overbought/oversold behavior
- **MACD, MACD signal, MACD histogram** — trend direction and momentum
- **High-Low range** — daily price range normalized by close
- **Market index features** — S&P 500 and Nasdaq lagged returns

**Lookahead bias prevention:** The target variable was shifted 5 days forward relative to all features. All features use only information available at prediction time.

**Data leakage fix:** An earlier version incorrectly included `Weekly_Return` as a feature, producing artificially high accuracy (99–100%). This was identified and corrected; all results use the cleaned feature set.

### 3.3 Target Variable

| Model | Target formulation |
|-------|-------------------|
| ARIMA | Regression — numeric 7-day return |
| XGBoost | 3-class — Down (<-1%), Neutral (±1%), Up (>+1%) |
| Elastic Net / Random Forest | Binary — positive (1) or negative (0) return |
| LSTM | 3-class — Down (<-1%), Neutral (±1%), Up (>+1%) |

The ±1% threshold for XGBoost and LSTM prevents small near-zero returns from being treated as meaningful signals. With 3 classes, the random baseline is 33% (not 50%).

### 3.4 Train/Validation/Test Split

Data was split **chronologically** to prevent lookahead bias. The held-out test period was **2024–2026** (approximately the last 20% of observations per stock).

- **ARIMA:** Walk-forward validation — retrain at every test step on all available history
- **XGBoost / Elastic Net / Random Forest:** `TimeSeriesSplit` (5-fold cross-validation)
- **LSTM:** Per-stock 80/20 chronological split, then pooled by sector

### 3.5 Model Details

**ARIMA / ARIMAX:**  
Walk-forward ARIMAX with order (1,1,1) applied to all 15 stocks. Exogenous variables: lagged returns, moving averages, volatility. Evaluated using MAE, RMSE, directional accuracy, and BUY/SELL backtesting.

**XGBoost — Corrected Multiclass Classifier:**  
The original version used `XGBRegressor` to predict numeric returns, then converted predictions to Up/Down using a zero cutoff. After feedback, this was corrected to `XGBClassifier` directly predicting three movement classes (Down, Neutral, Up) using a ±1% threshold. This avoids treating every near-zero return as a meaningful signal.

For stock ranking, predicted class probabilities are used:

**Ranking Score = Probability of Up − Probability of Down**

This considers both upside potential and downside risk. For each date, all 15 stocks are ranked by this score; the highest-ranked stock is selected for portfolio simulation.

Naive baselines included: always predict Up, always predict majority class. Evaluated using accuracy, **balanced accuracy**, macro precision, macro recall, **macro F1**, and confusion matrix. Balanced accuracy and macro F1 are emphasized because normal accuracy is misleading when one class dominates.

**Elastic Net Logistic Regression:**  
Binary classification (positive vs. negative return). L1+L2 regularization handles correlated features. Cross-validated using `TimeSeriesSplit`. Evaluated using accuracy, precision, recall, F1, confusion matrix.

**LSTM — Sector-Based 3-Class Classifier:**  
One LSTM per sector (Tech, Finance, Healthcare, Energy). Architecture: LSTM (16 units, L2 reg) → Dropout (0.3) → Dense (16, ReLU) → Softmax (3 classes). Balanced class weights counteract Up-majority bias. EarlyStopping patience=5.

**Random Forest:**  
Ensemble baseline using same feature set as XGBoost. Binary directional classifier. Cross-validated using `TimeSeriesSplit`.

### 3.6 Issues Encountered and Resolved

| Issue | Cause | Fix |
|-------|-------|-----|
| 99–100% accuracy in early models | Data leakage: `Weekly_Return` included as feature | Removed target-derived features; recomputed returns from raw `pct_change()` |
| XGBoost treating near-zero returns as signals | Regression → zero cutoff approach | Switched to `XGBClassifier` with ±1% threshold; added Neutral class |
| Only 5 stocks in LSTM results | Global time split on pooled sector data cut off most stocks | Split per stock first, then pool by sector |
| LSTM always predicting Up | Bull-market majority class bias | Added balanced class weights |
| Overfitting in LSTM | LSTM too large (32 units) | Reduced to 16 units, added L2 reg, EarlyStopping patience=5 |
| ARIMA on raw prices | Inflated MAE/RMSE | Fit ARIMA on weekly returns (`pct_change(5)`) instead |

---
## 4. Results

### 4.1 Model Evaluation Framework

All models evaluated on held-out test data (2024–2026).

**Cross-validation:** XGBoost, Elastic Net, and Random Forest used `TimeSeriesSplit` (5-fold). ARIMA used walk-forward validation. LSTM used chronological 80/20 per stock.

**Baselines:**
- Random selection: ~50% directional accuracy (binary), ~33% (3-class)
- Always predict Up baseline
- Majority class baseline
- Equal-weight buy-and-hold across all 15 stocks
- S&P 500 buy-and-hold

> **Additional results** (full confusion matrices, per-fold CV scores, feature importance plots) are in the supporting notebooks listed in Section 9.

### 4.2 ARIMA / ARIMAX Results

Walk-forward ARIMAX (1,1,1) applied to all 15 stocks. Results include MAE, RMSE, directional accuracy, and BUY/SELL backtesting simulation from $1,000 vs. buy-and-hold. Full per-stock results and backtesting curves are in `work/Models/Arima For 15 Stocks.ipynb`.

### 4.3 XGBoost Results — Corrected Multiclass

XGBoost was corrected from a regression-to-direction approach to a direct multiclass classifier predicting Down, Neutral, or Up. The corrected model:
- Uses `XGBClassifier` with ±1% threshold for the Neutral class
- Is evaluated using **balanced accuracy** and **macro F1** (not just accuracy) to avoid misleading results from class imbalance
- Is compared against naive baselines (always predict Up, majority class)
- Ranks stocks using **Probability of Up − Probability of Down**

Full per-stock metrics, feature importance, and portfolio simulation are in `work/Models/xgboost_15stocks_corrected_multiclass.ipynb` and the corresponding CSV files.

**Top features across all stocks:** RSI, MACD histogram, 5-day MA ratio, and 5-day lagged return were consistently the most important predictors.

### 4.4 Elastic Net Logistic Regression Results

Binary classification model producing direct buy/avoid signals for all 15 stocks. More interpretable than XGBoost but limited by the linearity assumption.

> Full accuracy, precision, recall, F1, and confusion matrices are in `work/Models/ElasticNet_Logistic_Model.ipynb`.

### 4.5 LSTM Results — Sector-Based 3-Class

All 15 stocks reported. **3-class random baseline = 33%.** Results above 40% are meaningfully above chance.

| Ticker | Sector | Accuracy | Down F1 | Neutral F1 | Up F1 | Macro F1 |
|--------|--------|----------|---------|------------|-------|----------|
| TSLA  | Tech        | 0.463 | 0.568 | 0.000 | 0.365 | 0.311 |
| BA    | Energy      | 0.436 | 0.383 | 0.103 | 0.556 | 0.347 |
| NVDA  | Tech        | 0.423 | 0.402 | 0.293 | 0.485 | 0.393 |
| PFE   | Healthcare  | 0.373 | 0.351 | 0.301 | 0.442 | 0.365 |
| WMT   | Energy      | 0.339 | 0.213 | 0.450 | 0.200 | 0.288 |
| JNJ   | Healthcare  | 0.336 | 0.117 | 0.505 | 0.178 | 0.267 |
| MSFT  | Tech        | 0.325 | 0.386 | 0.372 | 0.157 | 0.305 |
| JPM   | Finance     | 0.323 | 0.273 | 0.353 | 0.324 | 0.317 |
| GS    | Finance     | 0.321 | 0.245 | 0.337 | 0.357 | 0.313 |
| META  | Tech        | 0.307 | 0.423 | 0.216 | 0.214 | 0.284 |
| AMZN  | Tech        | 0.305 | 0.281 | 0.360 | 0.251 | 0.297 |
| CVX   | Energy      | 0.300 | 0.237 | 0.363 | 0.266 | 0.289 |
| XOM   | Energy      | 0.294 | 0.206 | 0.325 | 0.318 | 0.283 |
| AAPL  | Tech        | 0.282 | 0.194 | 0.385 | 0.189 | 0.256 |
| GOOGL | Tech        | 0.246 | 0.292 | 0.267 | 0.175 | 0.245 |

All 15 stocks beat the 33% random baseline. TSLA (46.3%), BA (43.6%), and NVDA (42.3%) were the strongest performers. Full confusion matrices and class distribution plots are in `work/Models/LSTM Model Improved.ipynb`.

### 4.6 Summary Comparison

| Model | Stocks | Task | Validation | Primary Metric | Output |
|-------|--------|------|------------|----------------|--------|
| ARIMA/ARIMAX | 15 | Return regression | Walk-forward | MAE + directional accuracy | BUY/SELL backtest |
| XGBoost | 15 | 3-class (Down/Neutral/Up) | TimeSeriesSplit 5-fold | Balanced accuracy, Macro F1 | Ranking: P(Up)−P(Down) |
| Elastic Net | 15 | Binary classification | TimeSeriesSplit 5-fold | Accuracy, F1 | Buy/Avoid signal |
| LSTM (sector) | 15 | 3-class (Down/Neutral/Up) | 80/20 per stock | Accuracy, Macro F1 | Down/Neutral/Up signal |
| Random Forest | 15 | Binary classification | TimeSeriesSplit 5-fold | Accuracy, F1 | Buy/Avoid signal |

---
## 5. Discussion

### 5.1 Were the Goals Achieved?

The project goal was largely achieved. The system produces a weekly ranked list of all 15 stocks by predicted return, generates buy/avoid signals, and simulates portfolio performance against a buy-and-hold benchmark. This directly addresses the core need of a student investor: a simple, actionable weekly output that requires no financial expertise to interpret.

The models did not all reach the 60% directional accuracy target set at the project's outset. XGBoost and the binary classification models (Elastic Net, Random Forest) came closest. The LSTM 3-class model achieved 25–46% accuracy, which, while below the binary target, is still above the 3-class random baseline of 33%. This is an honest outcome — weekly stock prediction is a genuinely hard problem, and the literature confirms that even professional models with rich feature sets rarely exceed 64% (Pinelis & Ruppert, 2022).

### 5.2 Connection to Stakeholder Need

For a student investor, the value of this system is not perfection — it is providing a data-driven starting point that is better than guessing. A student with no experience would otherwise pick stocks based on news headlines, social media, or gut feeling. Even a modest edge above random (52–59% directional accuracy from the binary models) translates into better outcomes over many weekly decisions.

The system's output is designed to be immediately usable: a ranked list with a buy/avoid label for each stock. The student does not need to understand LSTM or XGBoost — they just need the top-ranked stock and the expected direction. The portfolio simulation further contextualizes this by showing what a student starting with $1,000 and following the model's signals each week would have earned over the 2024–2026 test period.

### 5.3 LSTM Findings and Professor Feedback Integration

Based on professor feedback, three key improvements were made to the LSTM:

1. **3-class target** replacing binary classification to remove ambiguous near-zero moves
2. **Sector-based training** — one model per sector to capture shared dynamics within Tech, Finance, Healthcare, and Energy
3. **Balanced class weights** preventing the model from exploiting the Up-majority bias in bull-market training data

The sector-based approach confirmed the professor's hypothesis: stocks in the same sector share predictable patterns but are not uniform. TSLA and NVDA (high volatility/momentum tech) outperform AAPL and GOOGL (stable tech). Energy sector shows split behavior — BA and WMT perform better than XOM and CVX, suggesting industrial and consumer stocks have different short-term predictability than commodity-sensitive energy names.

The drop from v1 binary accuracy (52–59%) to v2 3-class accuracy (25–46%) is expected and not a regression: a 3-class problem is fundamentally harder. Both results are consistent with Pinelis & Ruppert (2022). The Neutral class collapse (model frequently predicts Neutral under uncertainty) is an honest finding: the ±1% neutral zone introduces ambiguity that technical features alone cannot resolve. Sentiment data would likely fix this.

---
## 6. Limitations

- **ARIMA scalability:** Walk-forward ARIMAX requires refitting at every test step, making it computationally prohibitive for large universes. Limited hyperparameter tuning was possible.
- **LSTM 3-class difficulty:** The ±1% neutral zone creates class imbalance in stable market periods, making Down and Up harder to detect. The model frequently defaults to Neutral under uncertainty.
- **XGBoost interpretability:** Captures nonlinear relationships but is less interpretable than simpler models. Feature importance helps but does not fully explain individual predictions.
- **Elastic Net linearity:** Assumes a mostly linear relationship between features and return probability. Market behavior is often more complex.
- **Feature scope:** All models rely primarily on historical price, volume, and market-based features. News events, earnings surprises, macroeconomic changes, and unexpected market shocks are not captured.
- **Sector definitions:** WMT and BA were grouped with Energy due to the 15-stock universe size. A larger universe would allow more precise sector groupings.

---
## 7. Future Work

- **Sentiment features:** Financial news sentiment scores or earnings call transcript embeddings would likely improve Neutral class detection in LSTM
- **Sector-specific macro features:** Oil prices for Energy, Fed funds rate for Finance, FDA approval cycles for Healthcare
- **Ensemble system:** Combine ARIMA, XGBoost, Elastic Net, LSTM, and Random Forest predictions via soft voting into a unified weekly ranking
- **Per-stock neutral threshold:** Adjust the ±1% boundary based on each stock's historical volatility rather than applying a uniform threshold
- **Transaction costs and risk adjustment:** Expand portfolio simulation to include transaction costs, stop-loss rules, Sharpe ratio, and better position sizing
- **Larger universe:** Extending to 50+ stocks would allow more meaningful sector groupings and a more realistic portfolio simulation

---
## 8. References

Fischer, T., & Krauss, C. (2018). Deep learning with long short-term memory networks for financial market predictions. *European Journal of Operational Research*, 270(2), 654–669.

Sezer, O. B., Gudelek, M. U., & Ozbayoglu, A. M. (2020). Financial time series forecasting with deep learning: A systematic literature review. *Applied Soft Computing*, 90, 106181.

Pinelis, M., & Ruppert, D. (2022). Machine learning portfolio allocation. *Journal of Financial Data Science*, 4(1), 117–129.

---
## 9. Supporting Notebook Index

*All supporting notebooks are in the `work/` folder. Data files are in `work/data/` (not committed to the repo — see Google Drive link in Section 1).*

| Notebook / File | Location | Purpose |
|---|---|---|
| `Arima For 15 Stocks.ipynb` | `work/Models/` | ARIMAX applied to all 15 stocks; walk-forward, per-stock MAE, RMSE, directional accuracy |
| `arima_model.ipynb` | `work/Models/` | ARIMA/ARIMAX on AAPL single stock; backtesting, BUY/SELL trading signals |
| `xgboost_15stocks_corrected_multiclass.ipynb` | `work/Models/` | Corrected XGBoost multiclass classifier (Down/Neutral/Up); balanced accuracy, macro F1, probability-based ranking |
| `ElasticNet_Logistic_Model.ipynb` | `work/Models/` | Elastic Net Logistic Regression — binary buy/avoid classification; accuracy, F1, confusion matrix |
| `Gradient_model.ipynb` | `work/Models/` | Gradient Boosting model experiments |
| `LSTM Model Improved.ipynb` | `work/Models/` | LSTM v2 — sector-based 3-class classifier, all 15 stocks; confusion matrices; class distribution plots |
| `LSTM_15model.ipynb` | `work/Models/` | LSTM applied to all 15 stocks (earlier version) |
| `LSTM_model.ipynb` | `work/Models/` | Initial LSTM model development and experiments |
| `Random_Forest.ipynb` | `work/Models/` | Random Forest binary classifier; feature importance; accuracy, F1 |
| `backtest_1.ipynb` | `work/` | Backtesting v1 — portfolio simulation using model signals |
| `backtest.ipynb` | `work/` | Backtesting — BUY/SELL signal simulation vs. buy-and-hold benchmark |
| `ranking_backtest.ipynb` | `work/` | Stock ranking backtest — weekly top-ranked stock selection and P&L simulation |
| EDA notebooks | `work/EDA/notebooks/` | Exploratory data analysis — return distributions, correlation heatmap, volatility comparison, class balance |
| `xgboost_multiclass_predictions_15_stocks.csv` | `work/data/results/` | XGBoost predictions with actual class, predicted class, and class probabilities for all 15 stocks |
| `xgboost_multiclass_metrics_15_stocks.csv` | `work/data/results/` | XGBoost evaluation metrics: accuracy, balanced accuracy, macro precision, recall, F1 |
| `xgboost_multiclass_feature_importance_15_stocks.csv` | `work/data/results/` | Feature importance from corrected XGBoost multiclass model |
| `xgboost_multiclass_top_ranked_portfolio.csv` | `work/data/results/` | Portfolio output based on XGBoost probability ranking score |
| `lstm_3class_predictions.csv` | `work/data/results/` | LSTM per-stock predictions with class probabilities (Down/Neutral/Up) |